# Energy Monitoring, Anomaly Detection & Root Cause Intelligence System

In [1]:
# CONFIG (Manual Tags)
TAGS = {
    "Motor current": "D1",
    "Motor voltage": "H1",
    "Output frequency": "B1"
}

TARGET_TAG = "D1"  # consumption source
METER_ID = "5073"
METER_NAME = "ONLINE00000090000003"

In [ ]:
# 2. Connect DB
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    "string"
)

In [5]:
# 2. LOAD + PIVOT DATA
query = f"""
SELECT RTC, TagId, Consumption
FROM RegisterDataInterval
WHERE EnergymeterMasterID = '{METER_NAME}'
AND TagId IN ({','.join([f"'{v}'" for v in TAGS.values()])})
ORDER BY RTC
"""

df = pd.read_sql(query, engine)

# Pivot
df_pivot = df.pivot(index='RTC', columns='TagId', values='Consumption')
df_pivot = df_pivot.rename(columns={v: k for k, v in TAGS.items()})

df_pivot = df_pivot.dropna().reset_index()
print(df_pivot.head(50))

TagId                 RTC  Output frequency  Motor current  Motor voltage
0     2025-11-14 08:00:00               0.0            0.0            0.0
1     2025-11-19 12:00:00               0.0            0.0            0.0
2     2025-11-19 12:15:00             250.0          100.0           40.0
3     2025-11-19 15:00:00            -225.0          -90.0            0.0
4     2025-11-19 16:00:00               0.0            0.0            0.0
5     2025-11-19 17:00:00               0.0            0.0            0.0
6     2025-11-21 08:00:00               0.0            0.0            0.0
7     2025-11-21 14:30:00               0.0            0.0            0.0
8     2025-11-24 12:00:00               0.0            0.0            0.0
9     2025-11-24 15:15:00               0.0            0.0            0.0
10    2025-11-26 08:00:00               0.0            0.0            0.0
11    2025-11-27 08:00:00               0.0            0.0            0.0
12    2025-11-28 08:00:00             

In [7]:
# 3. FEATURE ENGINEERING
df_pivot['RTC'] = pd.to_datetime(df_pivot['RTC'])

df_pivot['hour'] = df_pivot['RTC'].dt.hour
df_pivot['dayofweek'] = df_pivot['RTC'].dt.dayofweek

# target
df_pivot['target'] = df_pivot['Motor current']

# lag features
df_pivot['lag_1'] = df_pivot['target'].shift(1)
df_pivot['lag_2'] = df_pivot['target'].shift(2)

# rolling
df_pivot['rolling_mean'] = df_pivot['target'].rolling(3).mean()
df_pivot['rolling_std'] = df_pivot['target'].rolling(3).std()

df_pivot = df_pivot.dropna()
print(df_pivot.head(50))

TagId                 RTC  Output frequency  Motor current  Motor voltage  \
2     2025-11-19 12:15:00             250.0          100.0           40.0   
3     2025-11-19 15:00:00            -225.0          -90.0            0.0   
4     2025-11-19 16:00:00               0.0            0.0            0.0   
5     2025-11-19 17:00:00               0.0            0.0            0.0   
6     2025-11-21 08:00:00               0.0            0.0            0.0   
7     2025-11-21 14:30:00               0.0            0.0            0.0   
8     2025-11-24 12:00:00               0.0            0.0            0.0   
9     2025-11-24 15:15:00               0.0            0.0            0.0   
10    2025-11-26 08:00:00               0.0            0.0            0.0   
11    2025-11-27 08:00:00               0.0            0.0            0.0   
12    2025-11-28 08:00:00               0.0            0.0            0.0   

TagId  hour  dayofweek  target  lag_1  lag_2  rolling_mean  rolling_std  
2

In [9]:
# 4. TRAIN XGBOOST
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

features = [
    'Motor voltage', 'Output frequency',
    'hour', 'dayofweek',
    'lag_1', 'lag_2',
    'rolling_mean', 'rolling_std'
]

X = df_pivot[features]
y = df_pivot['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = XGBRegressor(n_estimators=300, max_depth=6)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [10]:
# ANOMALY DETECTION
import numpy as np

residuals = abs(y_test - preds)
threshold = np.percentile(residuals, 95)

def detect_anomaly(actual, predicted):
    residual = abs(actual - predicted)
    return residual > threshold, residual

In [ ]:
# 6. ROOT CAUSE
def explain_anomaly(row):
    reasons = []

    if row['Motor voltage'] > df_pivot['Motor voltage'].mean():
        reasons.append("High Voltage Spike")

    if row['Output frequency'] > df_pivot['Output frequency'].mean():
        reasons.append("Frequency fluctuation")

    if row['target'] > df_pivot['target'].mean() * 1.5:
        reasons.append("High Consumption Spike")

    if not reasons:
        reasons.append("Unexpected behavior")

    return reasons

In [12]:
# 7. COST INSIGHT
UNIT_RATE = 7  # or fetch from EnergyMeterMaster

def cost_insight(row):
    cost = row['target'] * UNIT_RATE

    if cost > df_pivot['target'].mean() * UNIT_RATE:
        return ["High energy cost → optimize usage"]

    return []

In [13]:
# GEMINI SUMMARY
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")

def summarize_with_llm(data):
    prompt = f"""
        Summarize energy anomaly:

        Prediction: {data['prediction']}
        Actual: {data['actual']}
        Reasons: {data['reasons']}
        Insights: {data['insights']}

        Give 2 line actionable insight.
    """
    
    res = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt
    )

    return res.text

In [14]:
# 9. PIPELINE
def pipeline(row):
    input_df = pd.DataFrame([row[features]]).astype(float)

    pred = model.predict(input_df)[0]
    flag, err = detect_anomaly(row['target'], pred)

    reasons = explain_anomaly(row) if flag else []
    insights = cost_insight(row)

    result = {
        "prediction": float(pred),
        "actual": float(row['target']),
        "anomaly": flag,
        "error": float(err),
        "reasons": reasons,
        "insights": insights
    }

    result["summary"] = summarize_with_llm(result)
    return result

In [17]:
# 10. RUN
valid_range = min(20, len(y_test))

start_idx = len(df_pivot) - len(y_test)

for i in range(-valid_range, 0):
    idx = start_idx + i
    
    if idx < 0 or idx >= len(df_pivot):
        continue  # skip invalid
    
    row = df_pivot.iloc[idx]
    print(pipeline(row))

{'prediction': -1.5258792700478807e-05, 'actual': 0.0, 'anomaly': np.False_, 'error': 1.5258792700478807e-05, 'reasons': [], 'insights': [], 'summary': '**Summary of Energy Anomaly:**\nNo significant energy anomaly was detected; the predicted energy value (-0.000015) was extremely close to the actual observed value (0.0). No specific reasons or insights were identified for this minimal deviation.\n\n**2-Line Actionable Insight:**\nThe model demonstrates high accuracy in predicting nominal energy states, confirming its reliability when no significant anomalies are present.\nFurther analyze these minuscule residuals to refine model sensitivity and potentially identify emergent micro-patterns that might precede larger events.'}
{'prediction': -1.5258792700478807e-05, 'actual': 0.0, 'anomaly': np.False_, 'error': 1.5258792700478807e-05, 'reasons': [], 'insights': [], 'summary': '**Summary of Energy Anomaly:**\nThe prediction for the energy anomaly was extremely close to zero (-1.53e-05), a